In [ ]:
import pandas as pd
df_inventory = pd.read_csv('CHU_Data_in_PSWS_Database.csv')
df_inventory.shape

In [ ]:
import io
import json
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

STATIONS_URL = "https://pswsnetwork.eng.ua.edu/stations/stations/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}


def scrape_station_grid_lookup(target_stations_list):
    """Scrapes the PSWS stations database to find Grid squares for our specific inventory stations."""
    # Convert list to a set for fast, O(1) matching lookups
    missing_stations = set(target_stations_list)
    grid_lookup_dict = {}

    current_page = 1
    print(f"Beginning Grid lookup for {len(missing_stations)} unique stations...")

    while missing_stations:
        print(f"Searching Stations Directory page {current_page}...")
        url = f"{STATIONS_URL}?page={current_page}"

        try:
            response = requests.get(url, headers=headers, timeout=12)

            # If we run past the final page of the directory, stop
            if response.status_code != 200:
                print(f"Reached directory boundary at page {current_page}.")
                break

            # FIX: Wrapped response.text in io.StringIO to resolve the deprecation warning
            tables = pd.read_html(io.StringIO(response.text))
            if not tables:
                print("No tables found. Ending lookup loop.")
                break

            # The station index table has columns: ['ID', 'User', 'Nickname', 'Grid', ...]
            df_page = tables[0]

            # Iterate through the rows on this directory page
            for _, row in df_page.iterrows():
                nickname = str(row.get("Nickname", "")).strip()
                grid = str(row.get("Grid", "")).strip()

                # If this station matches one we need, store its grid square
                if nickname in missing_stations:
                    # Filter out placeholders or blanks if any exist
                    if grid and grid != "—" and grid != "nan":
                        grid_lookup_dict[nickname] = grid
                        missing_stations.remove(nickname)

            # Look ahead in the pagination list to see if another page exists
            soup = BeautifulSoup(response.text, "html.parser")
            pagination_ul = soup.find("ul", class_="pagination")

            if pagination_ul:
                available_pages = [
                    int(a["href"].split("page=")[-1])
                    for a in pagination_ul.find_all("a")
                    if "page=" in a.get("href", "")
                ]
                if available_pages and (current_page + 1) > max(available_pages):
                    print("Scraped all available directory pages.")
                    break
            else:
                break

            current_page += 1
            time.sleep(1.0)  # Standard safety pause between pagination pages

        except Exception as e:
            print(f"Error reading directory page {current_page}: {e}")
            break

    print(
        f"\nLookup complete! Found grids for {len(grid_lookup_dict)} stations."
    )
    if missing_stations:
        print(f"Could not locate grid configurations for: {list(missing_stations)}")

    return grid_lookup_dict


# --- HOW TO RUN IT WITH YOUR EXISTING DATAFRAME ---
unique_inventory_stations = df_inventory["Station"].unique().tolist()
# (Mocking df_inventory just so the script remains syntactically executable)
# unique_inventory_stations = ["W2SJW", "N8XYZ"] 

# Run the dynamic scraper to fetch the grid codes
station_grid_lookup = scrape_station_grid_lookup(unique_inventory_stations)

print("\nGenerated Dictionary:")
print(station_grid_lookup)


# --- NEW: SAVE AND LOAD THE DICTIONARY LOCALLY ---
file_path = "station_grid_lookup.json"

# 1. Save the dictionary locally as a JSON file
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(station_grid_lookup, f, indent=4)
print(f"\nSuccessfully saved dictionary locally to '{file_path}'")

# 2. Read the dictionary back from the local file
with open(file_path, "r", encoding="utf-8") as f:
    loaded_grid_lookup = json.load(f)

print("Successfully reloaded dictionary from local file:")
print(loaded_grid_lookup)

In [ ]:
station_grid_lookup

In [ ]:
import maidenhead as mh
import pandas as pd
import plotly.express as px
df_inventory = pd.read_csv('CHU_Data_in_PSWS_Database.csv')

# 1. Ensure timestamps are proper datetime objects
df_map_timeline = df_inventory.copy()

# FIX: Remove the mismatching strict format string so pandas can correctly parse the space/colons
df_map_timeline["Start_UTC"] = pd.to_datetime(
    df_map_timeline["Start_UTC"], 
    errors="coerce"
)

# 2. Add a Month column (Outputs format: YYYY-MM)
df_map_timeline["Month"] = df_map_timeline["Start_UTC"].dt.to_period("M").astype(str)

# 3. Apply grid lookup dictionary to get coordinates
def grid_to_coordinates(station_name):
    grid = station_grid_lookup.get(str(station_name).strip(), None)
    if grid:
        try:
            lat, lon = mh.to_location(grid)
            return pd.Series([lat, lon])
        except Exception:
            return pd.Series([None, None])
    return pd.Series([None, None])

df_map_timeline[["Latitude", "Longitude"]] = df_map_timeline["Station"].apply(grid_to_coordinates)
df_map_timeline = df_map_timeline.dropna(subset=["Latitude", "Longitude"])


# --- EXPLODE & CLEAN FREQUENCIES FOR A PERFECT LEGEND ---

# Convert "3.330 MHz, 5.000 MHz" string arrays into clean python lists
df_map_timeline["Center_Frequencies"] = (
    df_map_timeline["Center_Frequencies"]
    .astype(str)
    .str.replace(" MHz", "", case=False) # Strip out unit text
    .str.split(", ")                     # Split strings into clean lists
)

# Break rows with lists into separate rows per frequency
df_map_timeline = df_map_timeline.explode("Center_Frequencies")
df_map_timeline["Center_Frequencies"] = df_map_timeline["Center_Frequencies"].str.strip()

# Filter strictly to our 3 targets so extra frequencies don't populate the map/legend
target_frequencies = ["3.330", "7.850", "14.670"]
df_map_timeline = df_map_timeline[df_map_timeline["Center_Frequencies"].isin(target_frequencies)]


# 4. Group the data by Month
df_counts_monthly = (
    df_map_timeline.groupby(["Month", "Station", "Latitude", "Longitude", "Instrument", "Center_Frequencies"])
    .agg(Observations_This_Month=("Observation_ID", "count"))
    .reset_index()
)


# --- THE MONTHLY PADDING FIX (REVISED) ---

# A. Get unique sets of your dimensions
all_months = df_counts_monthly["Month"].unique()
# Keep unique station-to-frequency mappings
station_metadata = df_counts_monthly[["Station", "Latitude", "Longitude", "Instrument", "Center_Frequencies"]].drop_duplicates()

# B. Create a master grid that includes Center_Frequencies to preserve your legend groups per frame
mux_monthly = pd.MultiIndex.from_product(
    [all_months, station_metadata["Station"], target_frequencies], 
    names=["Month", "Station", "Center_Frequencies"]
)
df_base_monthly = pd.DataFrame(index=mux_monthly).reset_index()

# C. Merge coordinates and instruments back onto our skeleton dataframe
station_geo = station_metadata[["Station", "Latitude", "Longitude", "Instrument"]].drop_duplicates()
df_padded_monthly = pd.merge(df_base_monthly, station_geo, on="Station", how="left")

# D. Merge our actual observation counts onto this monthly structure
df_animated_monthly = pd.merge(
    df_padded_monthly, 
    df_counts_monthly, 
    on=["Month", "Station", "Latitude", "Longitude", "Instrument", "Center_Frequencies"], 
    how="left"
).fillna({"Observations_This_Month": 0})


# --- EDIT: DYNAMIC SIZE MAPPING & PLOT LAYERING ---

# Dictionary mapping frequencies to custom marker sizes
# (Adjust these numbers to scale how large you want each frequency to be)
frequency_sizes = {
    "14.670": 3,
    "7.850": 9,
    "3.330": 14
}

def calculate_marker_size(row):
    if row["Observations_This_Month"] > 0:
        return frequency_sizes.get(row["Center_Frequencies"], 10)
    return 0

df_animated_monthly["Marker_Size"] = df_animated_monthly.apply(calculate_marker_size, axis=1)

# CRITICAL FIX FOR LAYER ORDERING:
# We sort by Month ascending so the animation progresses correctly, 
# and by Marker_Size DESCENDING so larger bubbles are rendered first, 
# ensuring smaller bubbles are always drawn cleanly on top.
df_animated_monthly = df_animated_monthly.sort_values(
    by=["Month", "Marker_Size"], 
    ascending=[True, False]
).reset_index(drop=True)


# 5. Build the Animated Map
fig = px.scatter_map(
    df_animated_monthly,
    lat="Latitude",
    lon="Longitude",
    color="Center_Frequencies", 
    size="Marker_Size",        
    size_max=20,  # Increased to match our maximum custom size map (14.670 MHz = 20)            
    hover_name="Station",
    
    hover_data={
        "Observations_This_Month": True, 
        "Instrument": True,
        "Marker_Size": False, 
        "Month": False
    },
    
# --- ANIMATION CONFIGURATION ---
    animation_frame="Month",  
    animation_group="Station",     
    
    # FIX: Reverse the frequency order here so Plotly processes the 
    # largest bubbles first, layering the smallest ones perfectly on top!
    category_orders={
        "Month": sorted(df_animated_monthly["Month"].unique()),
        "Center_Frequencies": ["3.330", "7.850", "14.670"]  # Big to Small
    },
    
    title="Growth of CHU Monitoring Network Over Time",
    zoom=2,
)

# 6. Layout Adjustments
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    title_x=0.5,
    map_style="open-street-map",
    height=650
)

# Smooth transition configuration for monthly playback
try:
    fig.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 600
    fig.layout.updatemenus[0].buttons[0].args[1]["transition"]["duration"] = 250
except (IndexError, KeyError):
    pass

fig.show()
fig.write_html('CHU_Monitoring_Stations_Over_Time.html')